# IJIES Technical Re-examination — 5-Model Classification Benchmark and Cross-Domain Evaluation

This notebook performs the reproducibility re-evaluation of the **classification branch only** on both the benchmark test set and the corrected cross-domain set.

**Models:** Custom CNN, EfficientNetB0, ResNet50V2, Vision Transformer (ViT), and YOLOv11m-cls.

**Detection models are intentionally excluded from this notebook.**


In [ ]:
# ============================================================
# 0. INSTALL + REPRODUCIBILITY ENVIRONMENT
# ============================================================
%pip install -q ultralytics timm

import os
import sys
import json
import time
import shutil
import hashlib
import zipfile
import subprocess
import gc
import platform
import marshal
import base64
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Python:", sys.version)

gpu_query = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=index,name,memory.total,memory.used",
        "--format=csv,noheader",
    ],
    capture_output=True,
    text=True,
    check=False,
)

print("\nGPU inventory:")
print(gpu_query.stdout.strip() or "No NVIDIA GPU reported")

gpus = []
t4s = []

for line in gpu_query.stdout.splitlines():
    if not line.strip():
        continue
    parts = [x.strip() for x in line.split(",")]
    if len(parts) >= 2:
        item = (int(parts[0]), parts[1])
        gpus.append(item)
        if "T4" in parts[1].upper():
            t4s.append(item)

if t4s:
    TEST_GPU_ID, TEST_GPU_NAME = t4s[0]
elif gpus:
    TEST_GPU_ID, TEST_GPU_NAME = gpus[0]
    print("WARNING: Tesla T4 not found; using", TEST_GPU_NAME)
else:
    raise RuntimeError(
        "No NVIDIA GPU detected. This editor-facing rerun was validated on Tesla T4."
    )

print("\nEvaluation GPU:", TEST_GPU_ID, TEST_GPU_NAME)


In [ ]:
# ============================================================
# 1. CONFIG
# ============================================================

BENCHMARK_ROOT_HINT = Path(
    "/kaggle/input/datasets/toilaxien/asl-benchmark-test-split"
)

CROSS_DOMAIN_ROOT_HINT = Path(
    "/kaggle/input/datasets/toilaxien/asl-test-cross-domain-datase/cross_domain_dataset"
)

MODEL_ROOT = Path(
    "/kaggle/input/datasets/toilaxien/asl-models-realtime-test"
)

WORK_ROOT = Path(
    "/kaggle/working/IJIES_EDITOR_CLASSIFICATION_BENCHMARK_CROSSDOMAIN_WORK"
)

EVIDENCE_ROOT = Path(
    "/kaggle/working/IJIES_EDITOR_CLASSIFICATION_BENCHMARK_CROSSDOMAIN_EVIDENCE"
)

# True = clean editor-facing rerun with no stale cached outputs.
# Set False only when resuming an interrupted run in the same Kaggle session.
RESET_OUTPUT = True

if RESET_OUTPUT:
    for p in [WORK_ROOT, EVIDENCE_ROOT]:
        if p.exists():
            shutil.rmtree(p)

WORK_ROOT.mkdir(parents=True, exist_ok=True)
EVIDENCE_ROOT.mkdir(parents=True, exist_ok=True)

# Keep the variable name expected by the verified helper/worker cells.
OUTPUT_ROOT = WORK_ROOT

IMG_SIZE = 224
BENCHMARK_EXPECTED_TOTAL = 17400
BENCHMARK_EXPECTED_PER_CLASS = 600
CROSS_DOMAIN_EXPECTED_PER_CLASS = 30
CROSS_DOMAIN_EXPECTED_TOTAL = 870

BATCH_SIZE = {
    "Custom_CNN": 64,
    "EfficientNetB0": 64,
    "ResNet50V2": 64,
    "ViT": 64,
    "YOLOv11m_cls": 64,
}

LABELS_EXPECTED = [
    "A","B","C","D","E","F","G","H","I","J","K","L","M","N",
    "O","P","Q","R","S","T","U","V","W","X","Y","Z",
    "delete","nothing","space",
]

AEMNST = {"A","E","M","N","S","T"}

MODEL_PATHS = {
    "Custom_CNN": MODEL_ROOT / "custom_cnn_asl.keras",
    "EfficientNetB0": MODEL_ROOT / "efficientnetb0_asl.keras",
    "ResNet50V2": MODEL_ROOT / "resnet50v2_asl.keras",
    "ViT": MODEL_ROOT / "ViT.pth",
    "YOLOv11m_cls": MODEL_ROOT / "YoLo11m_csf.pt",
}

# Classification branch only. Detection models are intentionally excluded.

CLASSIFICATION_MODELS = [
    "Custom_CNN",
    "EfficientNetB0",
    "ResNet50V2",
    "ViT",
    "YOLOv11m_cls",
]

DATASETS = {
    "benchmark": BENCHMARK_ROOT_HINT,
    "cross_domain": CROSS_DOMAIN_ROOT_HINT,
}

print("Benchmark :", BENCHMARK_ROOT_HINT)
print("Cross     :", CROSS_DOMAIN_ROOT_HINT)
print("Models    :", MODEL_ROOT)
print("Work      :", WORK_ROOT)
print("Evidence  :", EVIDENCE_ROOT)


In [ ]:
# ============================================================
# 2. HELPERS
# ============================================================

IMAGE_EXTS = {
    ".jpg",".jpeg",".png",".bmp",".webp"
}

def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(chunk_size),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        encoding="utf-8",
    )

def canonical_label(name):
    s = str(name).strip()

    if len(s) == 1:
        return s.upper()

    low = s.lower()

    # Dataset folder/model alias:
    # "del" is the same semantic class reported as "delete".
    if low == "del":
        return "delete"

    if low in {"delete","nothing","space"}:
        return low

    return s

def find_class_root(root):
    root = Path(root)

    if not root.exists():
        raise FileNotFoundError(root)

    candidates = []

    dirs = [root] + [
        p
        for p in root.rglob("*")
        if p.is_dir()
    ]

    for d in dirs:
        try:
            subdirs = [
                p
                for p in d.iterdir()
                if p.is_dir()
            ]
        except Exception:
            continue

        labels = {
            canonical_label(p.name)
            for p in subdirs
        }

        matches = len(
            labels.intersection(
                LABELS_EXPECTED
            )
        )

        if matches >= 20:
            candidates.append(
                (
                    matches,
                    len(subdirs),
                    len(str(d)),
                    d,
                )
            )

    if not candidates:
        raise RuntimeError(
            f"Không tìm thấy class-folder root trong {root}"
        )

    candidates.sort(
        key=lambda x: (
            -x[0],
            x[1],
            x[2],
        )
    )

    return candidates[0][3]

def image_size(path):
    try:
        with Image.open(path) as img:
            return int(img.width), int(img.height)
    except Exception:
        return None, None


def build_manifest(root):
    """
    Build the RAW image manifest exactly as it exists on disk.
    No image is silently removed here.
    """
    class_root = find_class_root(root)

    folder_names = sorted([
        p.name
        for p in class_root.iterdir()
        if p.is_dir()
        and canonical_label(p.name)
        in LABELS_EXPECTED
    ])

    class_labels = [
        canonical_label(x)
        for x in folder_names
    ]

    if len(class_labels) != 29:
        raise RuntimeError(
            f"Expected 29 classes, found {len(class_labels)}: {class_labels}"
        )

    class_to_idx = {
        label: idx
        for idx, label
        in enumerate(class_labels)
    }

    rows = []

    for folder_name in folder_names:
        folder = class_root / folder_name
        label = canonical_label(folder_name)

        for p in sorted(folder.rglob("*")):
            if (
                not p.is_file()
                or p.suffix.lower()
                not in IMAGE_EXTS
            ):
                continue

            w, h = image_size(p)

            rows.append({
                "absolute_path": str(p),
                "relative_path": str(
                    p.relative_to(class_root)
                ),
                "filename": p.name,
                "source_folder": folder_name,
                "label": label,
                "true_index": class_to_idx[label],
                "width": w,
                "height": h,
                "size_bytes": p.stat().st_size,
                "sha256": sha256_file(p),
            })

    manifest = pd.DataFrame(rows)

    return (
        class_root,
        class_labels,
        manifest,
    )


def normalize_cross_domain_to_published_870(
    raw_manifest,
    labels,
    expected_per_class=30,
):
    """
    Reconstruct the published cross-domain evaluation size:
        29 classes × 30 images = 870 images.

    Policy:
    1. Never accept a class with fewer than 30 images.
    2. If a class has >30 images:
       a) first remove redundant exact-SHA256 copies, keeping one copy;
       b) if >30 still remain, keep the first 30 by relative_path
          deterministically and record every excluded file.
    3. Save an exclusion audit so the 870-image subset is transparent.

    Important:
    A lexicographic trim does NOT prove which file was absent from the
    historical 870-image evaluation. It only creates a deterministic
    30-per-class rerun when the current Kaggle folder contains an extra image.
    """
    if raw_manifest.empty:
        raise RuntimeError(
            "Cross-domain raw manifest is empty."
        )

    selected_parts = []
    excluded_parts = []
    audit_rows = []

    for label in labels:
        cls = (
            raw_manifest[
                raw_manifest["label"] == label
            ]
            .copy()
            .sort_values(
                ["relative_path", "sha256"]
            )
            .reset_index(drop=True)
        )

        n_raw = len(cls)

        if n_raw < expected_per_class:
            raise RuntimeError(
                f"Cross-domain class '{label}' has only {n_raw} images; "
                f"expected at least {expected_per_class}."
            )

        excluded_for_label = []

        # ------------------------------------------------------
        # Step A: exact-duplicate reduction, only if class > 30
        # ------------------------------------------------------
        work = cls.copy()

        if len(work) > expected_per_class:
            dup_mask = work.duplicated(
                subset=["sha256"],
                keep="first"
            )

            dup_candidates = work[dup_mask].copy()

            # Remove only as many exact duplicate copies as needed.
            excess = len(work) - expected_per_class

            if excess > 0 and not dup_candidates.empty:
                remove_dup = dup_candidates.head(excess).copy()
                remove_dup["exclusion_reason"] = (
                    "exact_sha256_duplicate_extra_copy"
                )

                excluded_for_label.append(
                    remove_dup
                )

                work = work.drop(
                    index=remove_dup.index
                ).reset_index(drop=True)

        # ------------------------------------------------------
        # Step B: if still >30, deterministic lexicographic trim
        # ------------------------------------------------------
        if len(work) > expected_per_class:
            keep = work.head(
                expected_per_class
            ).copy()

            remove_extra = work.iloc[
                expected_per_class:
            ].copy()

            remove_extra["exclusion_reason"] = (
                "deterministic_lexicographic_trim_to_30"
            )

            excluded_for_label.append(
                remove_extra
            )

            work = keep

        if len(work) != expected_per_class:
            raise RuntimeError(
                f"Cross-domain class '{label}' normalized to "
                f"{len(work)} images instead of {expected_per_class}."
            )

        selected_parts.append(
            work
        )

        if excluded_for_label:
            excluded_cls = pd.concat(
                excluded_for_label,
                ignore_index=True,
            )

            excluded_parts.append(
                excluded_cls
            )

            excluded_count = len(
                excluded_cls
            )

            exclusion_reasons = ";".join(
                sorted(
                    excluded_cls[
                        "exclusion_reason"
                    ].astype(str).unique()
                )
            )
        else:
            excluded_count = 0
            exclusion_reasons = ""

        audit_rows.append({
            "label": label,
            "raw_count": int(n_raw),
            "selected_count": int(len(work)),
            "excluded_count": int(excluded_count),
            "exclusion_reasons": exclusion_reasons,
        })

    selected = pd.concat(
        selected_parts,
        ignore_index=True,
    )

    excluded = (
        pd.concat(
            excluded_parts,
            ignore_index=True,
        )
        if excluded_parts
        else
        pd.DataFrame(
            columns=list(
                raw_manifest.columns
            ) + [
                "exclusion_reason"
            ]
        )
    )

    audit = pd.DataFrame(
        audit_rows
    )

    expected_total = (
        len(labels)
        * expected_per_class
    )

    if len(selected) != expected_total:
        raise RuntimeError(
            f"Cross-domain normalized total = {len(selected)}, "
            f"expected {expected_total}."
        )

    selected_counts = (
        selected.groupby("label")
        .size()
        .reindex(labels)
    )

    if not (
        selected_counts
        == expected_per_class
    ).all():
        raise RuntimeError(
            "Cross-domain normalization failed: "
            "not every class has exactly 30 images."
        )

    return (
        selected,
        excluded,
        audit,
    )


def run_child(name, cmd, timeout=None):
    env = os.environ.copy()

    env["CUDA_VISIBLE_DEVICES"] = str(TEST_GPU_ID)
    env["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
    env["TF_CPP_MIN_LOG_LEVEL"] = "2"
    env["TF_ENABLE_ONEDNN_OPTS"] = "0"
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    print("\n" + "="*80)
    print("START:", name)
    print("="*80)

    t0 = time.perf_counter()

    try:
        p = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            env=env,
            timeout=timeout,
            check=False,
        )

        rc = p.returncode
        stdout = p.stdout
        stderr = p.stderr

    except subprocess.TimeoutExpired as e:
        rc = "TIMEOUT"
        stdout = e.stdout or ""
        stderr = e.stderr or ""

    elapsed = time.perf_counter() - t0

    print("returncode:", rc)
    print("wall_seconds:", round(elapsed, 2))

    tail = stderr if stderr else stdout

    if tail:
        print(tail[-4000:])

    gc.collect()
    time.sleep(2)

    return {
        "name": name,
        "returncode": rc,
        "wall_seconds": elapsed,
        "stdout": stdout,
        "stderr": stderr,
    }


## 2. Dataset manifests and the 870-image cross-domain protocol

The benchmark test must contain exactly **17,400 images** (600 per class).

For the cross-domain set, the notebook audits every image currently present on
disk, then uses exactly **30 images per class**. If the current folder contains
an extra image, the notebook first prefers removal of redundant exact-SHA256
copies; if the excess is not an exact duplicate, it applies a deterministic
lexicographic trim and records the exclusion.

This is a transparent post-publication reconstruction. It does **not** claim
that a deterministic trim proves the identity of the historical 870-image
subset.


In [ ]:
# ============================================================
# 3. BUILD BOTH FINAL MANIFESTS — STRICT, NO AUTO-TRIMMING
# ============================================================

dataset_info = {}

for dataset_name, root_hint in DATASETS.items():
    root, labels, manifest = build_manifest(
        root_hint
    )

    if manifest.empty:
        raise RuntimeError(
            f"{dataset_name}: empty dataset"
        )

    ds_dir = (
        OUTPUT_ROOT
        / "datasets"
        / dataset_name
    )

    ds_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    expected_total = (
        BENCHMARK_EXPECTED_TOTAL
        if dataset_name == "benchmark"
        else CROSS_DOMAIN_EXPECTED_TOTAL
    )

    expected_per_class = (
        BENCHMARK_EXPECTED_PER_CLASS
        if dataset_name == "benchmark"
        else CROSS_DOMAIN_EXPECTED_PER_CLASS
    )

    if len(manifest) != expected_total:
        raise RuntimeError(
            f"{dataset_name}: expected {expected_total} images, "
            f"found {len(manifest)}. "
            "Use the final corrected dataset before continuing."
        )

    distribution = (
        manifest
        .groupby("label")
        .size()
        .reindex(labels)
        .rename("n_images")
        .reset_index()
    )

    if not (
        distribution["n_images"]
        == expected_per_class
    ).all():
        raise RuntimeError(
            f"{dataset_name}: expected exactly "
            f"{expected_per_class} images per class."
        )

    duplicate_rows = (
        manifest[
            manifest.duplicated(
                "sha256",
                keep=False,
            )
        ]
        .sort_values(
            "sha256"
        )
    )

    if dataset_name == "cross_domain" and not duplicate_rows.empty:
        raise RuntimeError(
            "Cross-domain set contains exact internal duplicates."
        )

    manifest_csv = (
        ds_dir
        / "manifest.csv"
    )

    manifest.to_csv(
        manifest_csv,
        index=False
    )

    distribution.to_csv(
        ds_dir
        / "class_distribution.csv",
        index=False
    )

    duplicate_rows.to_csv(
        ds_dir
        / "exact_duplicates.csv",
        index=False
    )

    audit = {
        "resolved_class_root":
            str(root),

        "n_classes":
            int(
                manifest["label"].nunique()
            ),

        "n_images":
            int(
                len(manifest)
            ),

        "expected_images":
            int(
                expected_total
            ),

        "expected_per_class":
            int(
                expected_per_class
            ),

        "min_images_per_class":
            int(
                distribution["n_images"].min()
            ),

        "max_images_per_class":
            int(
                distribution["n_images"].max()
            ),

        "exact_duplicate_rows":
            int(
                len(duplicate_rows)
            ),

        "exact_duplicate_sha256_groups":
            int(
                duplicate_rows["sha256"].nunique()
            ),
    }

    save_json(
        audit,
        ds_dir
        / "dataset_audit.json"
    )

    dataset_info[
        dataset_name
    ] = {
        "root":
            root,

        "labels":
            labels,

        "manifest":
            manifest,

        "manifest_csv":
            manifest_csv,

        "audit":
            audit,
    }

    print(
        "\n",
        dataset_name,
        json.dumps(
            audit,
            indent=2,
            ensure_ascii=False,
        )
    )

BENCH_LABELS = dataset_info[
    "benchmark"
]["labels"]

CROSS_LABELS = dataset_info[
    "cross_domain"
]["labels"]

if BENCH_LABELS != CROSS_LABELS:
    raise RuntimeError(
        "Benchmark and cross-domain class order differ."
    )

CLASS_LABELS = BENCH_LABELS

CLASS_MAPPING_CSV = (
    OUTPUT_ROOT
    / "class_mapping.csv"
)

pd.DataFrame({
    "class_index":
        range(
            len(CLASS_LABELS)
        ),

    "class_name":
        CLASS_LABELS,
}).to_csv(
    CLASS_MAPPING_CSV,
    index=False,
)

print(
    "\n✅ Strict final datasets verified."
)

print(
    "Benchmark:",
    len(
        dataset_info[
            "benchmark"
        ][
            "manifest"
        ]
    )
)

print(
    "Cross-domain:",
    len(
        dataset_info[
            "cross_domain"
        ][
            "manifest"
        ]
    )
)

print(
    "Shared class order:",
    CLASS_LABELS
)


In [ ]:
# ============================================================
# 4. EXACT BENCHMARK ↔ CROSS-DOMAIN OVERLAP
# ============================================================

benchmark_manifest = dataset_info[
    "benchmark"
]["manifest"]

cross_manifest = dataset_info[
    "cross_domain"
]["manifest"]

overlap = (
    benchmark_manifest[
        [
            "sha256",
            "relative_path",
            "label",
        ]
    ]
    .merge(
        cross_manifest[
            [
                "sha256",
                "relative_path",
                "label",
            ]
        ],
        on="sha256",
        suffixes=(
            "_benchmark",
            "_cross"
        ),
    )
)

overlap.to_csv(
    OUTPUT_ROOT
    / "benchmark_cross_exact_overlap.csv",
    index=False
)

save_json(
    {
        "benchmark_images":
            int(len(benchmark_manifest)),

        "cross_domain_images":
            int(len(cross_manifest)),

        "exact_overlap_pairs":
            int(len(overlap)),

        "note":
            (
                "This checks only benchmark-test vs cross-domain exact SHA256 overlap. "
                "It does not replace train/validation/test leakage audit."
            ),
    },
    OUTPUT_ROOT
    / "benchmark_cross_overlap_audit.json"
)

print(
    "Exact benchmark-cross overlap pairs:",
    len(overlap)
)


# ------------------------------------------------------------
# FINAL CLEAN-DATASET ASSERTION
# ------------------------------------------------------------
if len(overlap) != 0:
    raise RuntimeError(
        f"Final cross-domain dataset still overlaps benchmark: "
        f"{len(overlap)} exact SHA256 pairs."
    )

print("✅ Final benchmark↔cross-domain exact overlap: 0")


In [ ]:
# ============================================================
# 5. MODEL INVENTORY + CHECKPOINT HASHES
# ============================================================

inventory = []

for name, path in MODEL_PATHS.items():
    p = Path(path)
    inventory.append({
        "model": name,
        "source_filename": p.name,
        "exists": bool(p.exists()),
        "size_mb": (
            p.stat().st_size / 1024**2
            if p.exists()
            else None
        ),
        "sha256": (
            sha256_file(p)
            if p.exists()
            else None
        ),
    })

model_inventory = pd.DataFrame(inventory)
model_inventory.to_csv(
    WORK_ROOT / "model_inventory.csv",
    index=False
)

display(model_inventory)

missing = model_inventory.loc[
    ~model_inventory["exists"], "model"
].tolist()

if missing:
    raise FileNotFoundError(
        f"Missing model checkpoints: {missing}"
    )


## 6. EfficientNetB0 Keras-3 runtime compatibility audit

The original `.keras` checkpoint contains a serialized no-weight preprocessing
Lambda. For current Keras 3 runtime compatibility, this notebook reconstructs
only that preprocessing layer with the equivalent operation
`preprocess_input(x * 255.0)`. The original model weights are not modified.
Both original and compatibility artifacts are SHA256-audited.


In [ ]:
# ============================================================
# 6. BUILD EFFICIENTNET COMPAT ARCHIVE
# ============================================================

EFF_ORIGINAL = MODEL_PATHS[
    "EfficientNetB0"
]

EFF_COMPAT = Path(
    str(WORK_ROOT / "efficientnetb0_asl_kaggle_compat_benchmark_cross.keras")
)

EFF_TEMP = Path(
    str(WORK_ROOT / "effnet_benchmark_cross_compat_build")
)

if EFF_TEMP.exists():
    shutil.rmtree(EFF_TEMP)

if EFF_COMPAT.exists():
    EFF_COMPAT.unlink()

EFF_TEMP.mkdir(
    parents=True,
    exist_ok=True
)

with zipfile.ZipFile(
    EFF_ORIGINAL,
    "r"
) as z:
    z.extractall(EFF_TEMP)

config_path = EFF_TEMP / "config.json"

config = json.loads(
    config_path.read_text(
        encoding="utf-8"
    )
)

target = None

for layer in config[
    "config"
]["layers"]:
    if (
        layer.get("class_name") == "Lambda"
        and
        layer.get(
            "config",
            {}
        ).get("name") == "effnet_preprocess"
    ):
        target = layer
        break

if target is None:
    raise RuntimeError(
        "effnet_preprocess Lambda not found."
    )

fn = target[
    "config"
].get("function", {})

fn_cfg = (
    fn.get("config", {})
    if isinstance(fn, dict)
    else {}
)

code_b64 = fn_cfg.get("code")

audit = {
    "layer_name":
        target["config"].get("name"),

    "serialized_code_present":
        bool(code_b64),

    "original_model_sha256":
        sha256_file(EFF_ORIGINAL),
}

if code_b64:
    try:
        code_obj = marshal.loads(
            base64.b64decode(
                code_b64
            )
        )

        audit["co_names"] = list(
            code_obj.co_names
        )

        audit["co_consts"] = [
            repr(x)
            for x in code_obj.co_consts
        ]

        print(
            "Lambda co_names:",
            code_obj.co_names
        )

        print(
            "Lambda co_consts:",
            code_obj.co_consts
        )

        if (
            "preprocess_input"
            not in code_obj.co_names
        ):
            raise RuntimeError(
                "Lambda does not reference preprocess_input. Stop."
            )

    except RuntimeError:
        raise

    except Exception as e:
        audit[
            "bytecode_decode_error"
        ] = repr(e)

weights_candidates = sorted(
    EFF_TEMP.glob(
        "*.weights.h5"
    )
)

if weights_candidates:
    audit["weights_filename"] = (
        weights_candidates[0].name
    )

    audit["weights_sha256"] = (
        sha256_file(
            weights_candidates[0]
        )
    )

old_cfg = target["config"]

target["module"] = None
target["class_name"] = (
    "EffNetPreprocessCompat"
)
target["registered_name"] = (
    "EffNetPreprocessCompat"
)

target["config"] = {
    "name":
        old_cfg.get(
            "name",
            "effnet_preprocess"
        ),

    "trainable":
        old_cfg.get(
            "trainable",
            True
        ),

    "dtype":
        old_cfg.get(
            "dtype",
            "float32"
        ),
}

config_path.write_text(
    json.dumps(config),
    encoding="utf-8"
)

with zipfile.ZipFile(
    EFF_COMPAT,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for p in EFF_TEMP.rglob("*"):
        if p.is_file():
            z.write(
                p,
                p.relative_to(EFF_TEMP)
            )

audit.update({
    "compat_model_path":
        str(EFF_COMPAT),

    "compat_model_sha256":
        sha256_file(EFF_COMPAT),

    "weights_modified":
        False,

    "replacement":
        (
            "EffNetPreprocessCompat implementing "
            "preprocess_input(x*255.0)"
        ),

    "status":
        "POST_PUBLICATION_RUNTIME_COMPATIBILITY_RECONSTRUCTION",
})

save_json(
    audit,
    OUTPUT_ROOT
    / "efficientnet_compatibility_audit.json"
)

MODEL_PATHS[
    "EfficientNetB0"
] = EFF_COMPAT

print(
    "✅ EfficientNet compatibility archive ready:"
)

print(EFF_COMPAT)


## 8. Write isolated classifier worker


In [ ]:
# ============================================================
# 7. WRITE ISOLATED WORKERS
# ============================================================

CLS_WORKER_PATH = WORK_ROOT / "ijies_benchmark_cross_classifier_worker.py"

CLS_WORKER_PATH.write_text(
    '\nimport os\nimport sys\nimport json\nimport time\nfrom pathlib import Path\n\nMODEL_NAME = sys.argv[1]\nMODEL_PATH = Path(sys.argv[2])\nMANIFEST_CSV = Path(sys.argv[3])\nCLASS_MAPPING_CSV = Path(sys.argv[4])\nOUT_DIR = Path(sys.argv[5])\nIMG_SIZE = int(sys.argv[6])\nBATCH_SIZE = int(sys.argv[7])\n\nimport random\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\nrandom.seed(42)\nnp.random.seed(42)\n\nmanifest = pd.read_csv(MANIFEST_CSV)\nclass_map = pd.read_csv(CLASS_MAPPING_CSV)\n\nCLASS_LABELS = (\n    class_map\n    .sort_values("class_index")["class_name"]\n    .astype(str)\n    .tolist()\n)\n\nOUT_DIR.mkdir(parents=True, exist_ok=True)\n\npartial_csv = OUT_DIR / "predictions.partial.csv"\nfinal_csv = OUT_DIR / "predictions.csv"\nmeta_json = OUT_DIR / "worker_metadata.json"\n\ndef canonical_label(name):\n    s = str(name).strip()\n\n    if len(s) == 1:\n        return s.upper()\n\n    low = s.lower()\n\n    # "del" and "delete" are the same semantic class.\n    if low == "del":\n        return "delete"\n\n    if low in {"delete", "nothing", "space"}:\n        return low\n\n    return s\n\n# ------------------------------------------------------------\n# Resume\n# ------------------------------------------------------------\nrows = []\ndone = set()\n\nif partial_csv.exists():\n    old = pd.read_csv(partial_csv)\n\n    required = {\n        "relative_path",\n        "true_label",\n        "pred_label",\n        "correct",\n    }\n\n    if required.issubset(old.columns):\n        rows = old.to_dict("records")\n        done = set(old["relative_path"].astype(str))\n        print("Resume images:", len(done))\n    else:\n        print("Ignoring incompatible partial CSV.")\n\nremaining = manifest[\n    ~manifest["relative_path"].astype(str).isin(done)\n].copy()\n\n# ------------------------------------------------------------\n# Model load\n# ------------------------------------------------------------\nframework = None\nmodel = None\nload_t0 = time.perf_counter()\n\nif MODEL_NAME in {"Custom_CNN", "EfficientNetB0", "ResNet50V2"}:\n    import tensorflow as tf\n    from tensorflow import keras\n\n    gpus = tf.config.list_physical_devices("GPU")\n\n    if gpus:\n        try:\n            tf.config.experimental.set_memory_growth(gpus[0], True)\n        except RuntimeError:\n            pass\n\n    class Cast(tf.keras.layers.Layer):\n        def __init__(self, dtype="float32", **kwargs):\n            super().__init__(**kwargs)\n            self._dtype = dtype\n\n        def call(self, inputs):\n            return tf.cast(inputs, self._dtype)\n\n        def get_config(self):\n            cfg = super().get_config()\n            cfg.update({"dtype": self._dtype})\n            return cfg\n\n    class EffNetPreprocessCompat(tf.keras.layers.Layer):\n        """\n        Keras-3-safe equivalent of historical:\n            lambda z: preprocess_input(z * 255.0)\n        No trainable weights.\n        """\n        def __init__(self, **kwargs):\n            super().__init__(**kwargs)\n            self.supports_masking = True\n\n        def call(self, inputs, training=None, mask=None, **kwargs):\n            x = inputs * tf.cast(255.0, inputs.dtype)\n            return tf.keras.applications.efficientnet.preprocess_input(x)\n\n        def compute_mask(self, inputs, mask=None):\n            return mask\n\n        def get_config(self):\n            return super().get_config()\n\n    custom_objects = {\n        "Cast": Cast,\n    }\n\n    if MODEL_NAME == "EfficientNetB0":\n        custom_objects["EffNetPreprocessCompat"] = EffNetPreprocessCompat\n\n    model = keras.models.load_model(\n        str(MODEL_PATH),\n        custom_objects=custom_objects,\n        compile=False,\n        safe_mode=False,\n    )\n\n    print("Loaded:", MODEL_NAME)\n    print("Input :", model.input_shape)\n    print("Output:", model.output_shape)\n\n    if model.output_shape[-1] != 29:\n        raise RuntimeError(\n            f"{MODEL_NAME}: expected 29 outputs, got {model.output_shape}"\n        )\n\n    if MODEL_NAME == "EfficientNetB0":\n        dummy = tf.zeros((1, 224, 224, 3), dtype=tf.float32)\n        smoke = model(dummy, training=False).numpy()\n\n        if smoke.shape != (1, 29):\n            raise RuntimeError(\n                f"EfficientNet smoke output shape: {smoke.shape}"\n            )\n\n        if not np.isfinite(smoke).all():\n            raise RuntimeError(\n                "EfficientNet smoke output contains NaN/Inf."\n            )\n\n        print("EfficientNet dummy forward: PASS")\n\n    framework = "keras"\n\nelif MODEL_NAME == "ViT":\n    import torch\n    import timm\n    from torchvision import transforms\n\n    device = "cuda:0" if torch.cuda.is_available() else "cpu"\n\n    model = timm.create_model(\n        "vit_base_patch16_224",\n        pretrained=False,\n        num_classes=29,\n    )\n\n    state_dict = torch.load(\n        str(MODEL_PATH),\n        map_location="cpu",\n    )\n\n    if isinstance(state_dict, dict) and "state_dict" in state_dict:\n        state_dict = state_dict["state_dict"]\n\n    if any(str(k).startswith("module.") for k in state_dict.keys()):\n        state_dict = {\n            str(k).replace("module.", "", 1): v\n            for k, v in state_dict.items()\n        }\n\n    model.load_state_dict(\n        state_dict,\n        strict=True,\n    )\n\n    model.eval().to(device)\n\n    transform = transforms.Compose([\n        transforms.Resize((224, 224)),\n        transforms.ToTensor(),\n        transforms.Normalize(\n            mean=[0.5, 0.5, 0.5],\n            std=[0.5, 0.5, 0.5],\n        ),\n    ])\n\n    framework = "vit"\n\nelif MODEL_NAME == "YOLOv11m_cls":\n    import torch\n    from ultralytics import YOLO\n\n    model = YOLO(str(MODEL_PATH))\n\n    if len(model.names) != 29:\n        raise RuntimeError(\n            f"YOLOv11m-cls expected 29 classes, got {len(model.names)}"\n        )\n\n    test_names = {\n        x.lower(): i\n        for i, x in enumerate(CLASS_LABELS)\n    }\n\n    for _, name in model.names.items():\n        label = canonical_label(name)\n\n        if label.lower() not in test_names:\n            raise RuntimeError(\n                f"YOLO class \'{name}\' not present in dataset mapping."\n            )\n\n    framework = "yolo_cls"\n\nelse:\n    raise ValueError(MODEL_NAME)\n\nload_seconds = time.perf_counter() - load_t0\n\n# ------------------------------------------------------------\n# Helpers\n# ------------------------------------------------------------\ndef load_keras_image(path):\n    img = Image.open(path).convert("RGB")\n    img = img.resize((IMG_SIZE, IMG_SIZE))\n    return np.asarray(img, dtype=np.float32) / 255.0\n\ndef checkpoint(batch_rows):\n    global rows\n    rows.extend(batch_rows)\n    pd.DataFrame(rows).to_csv(partial_csv, index=False)\n\nrecords = remaining.to_dict("records")\n\n# ------------------------------------------------------------\n# Keras inference\n# ------------------------------------------------------------\nif framework == "keras":\n    import tensorflow as tf\n\n    input_name = model.inputs[0].name.split(":")[0]\n\n    for start in range(0, len(records), BATCH_SIZE):\n        batch = records[start:start+BATCH_SIZE]\n\n        x = np.stack([\n            load_keras_image(r["absolute_path"])\n            for r in batch\n        ])\n\n        t0 = time.perf_counter()\n\n        if MODEL_NAME == "EfficientNetB0":\n            probs = model(\n                {input_name: tf.convert_to_tensor(x)},\n                training=False,\n            ).numpy()\n        else:\n            probs = model(\n                tf.convert_to_tensor(x),\n                training=False,\n            ).numpy()\n\n        elapsed_ms = (time.perf_counter() - t0) * 1000.0\n\n        pred_idx = np.argmax(probs, axis=1)\n        conf = np.max(probs, axis=1)\n\n        batch_rows = []\n\n        for r, pi, cf in zip(batch, pred_idx, conf):\n            pi = int(pi)\n            pred_label = CLASS_LABELS[pi]\n\n            batch_rows.append({\n                "relative_path": r["relative_path"],\n                "absolute_path": r["absolute_path"],\n                "true_label": r["label"],\n                "true_index": int(r["true_index"]),\n                "pred_index": pi,\n                "pred_label": pred_label,\n                "confidence": float(cf),\n                "correct": pred_label == r["label"],\n                "batch_elapsed_ms": float(elapsed_ms),\n                "batch_mean_inference_ms_per_image": float(\n                    elapsed_ms / max(1, len(batch))\n                ),\n            })\n\n        checkpoint(batch_rows)\n\n        print(\n            MODEL_NAME,\n            min(start + len(batch), len(records)),\n            "/",\n            len(records),\n        )\n\n# ------------------------------------------------------------\n# ViT inference\n# ------------------------------------------------------------\nelif framework == "vit":\n    import torch\n\n    for start in range(0, len(records), BATCH_SIZE):\n        batch = records[start:start+BATCH_SIZE]\n\n        tensors = []\n\n        for r in batch:\n            img = Image.open(r["absolute_path"]).convert("RGB")\n            tensors.append(transform(img))\n\n        x = torch.stack(tensors, dim=0).to(device)\n\n        if torch.cuda.is_available():\n            torch.cuda.synchronize()\n\n        t0 = time.perf_counter()\n\n        with torch.inference_mode():\n            logits = model(x)\n            probs = torch.softmax(logits, dim=1)\n\n        if torch.cuda.is_available():\n            torch.cuda.synchronize()\n\n        elapsed_ms = (time.perf_counter() - t0) * 1000.0\n\n        conf, pred_idx = probs.max(dim=1)\n\n        conf = conf.detach().cpu().numpy()\n        pred_idx = pred_idx.detach().cpu().numpy()\n\n        batch_rows = []\n\n        for r, pi, cf in zip(batch, pred_idx, conf):\n            pi = int(pi)\n            pred_label = CLASS_LABELS[pi]\n\n            batch_rows.append({\n                "relative_path": r["relative_path"],\n                "absolute_path": r["absolute_path"],\n                "true_label": r["label"],\n                "true_index": int(r["true_index"]),\n                "pred_index": pi,\n                "pred_label": pred_label,\n                "confidence": float(cf),\n                "correct": pred_label == r["label"],\n                "batch_elapsed_ms": float(elapsed_ms),\n                "batch_mean_inference_ms_per_image": float(\n                    elapsed_ms / max(1, len(batch))\n                ),\n            })\n\n        checkpoint(batch_rows)\n\n        print(\n            MODEL_NAME,\n            min(start + len(batch), len(records)),\n            "/",\n            len(records),\n        )\n\n# ------------------------------------------------------------\n# YOLO classification inference\n# ------------------------------------------------------------\nelse:\n    import torch\n\n    class_to_idx = {\n        x.lower(): i\n        for i, x in enumerate(CLASS_LABELS)\n    }\n\n    for start in range(0, len(records), BATCH_SIZE):\n        batch = records[start:start+BATCH_SIZE]\n        paths = [r["absolute_path"] for r in batch]\n\n        if torch.cuda.is_available():\n            torch.cuda.synchronize()\n\n        t0 = time.perf_counter()\n\n        results = model.predict(\n            source=paths,\n            imgsz=IMG_SIZE,\n            batch=BATCH_SIZE,\n            verbose=False,\n            device=0,\n        )\n\n        if torch.cuda.is_available():\n            torch.cuda.synchronize()\n\n        elapsed_ms = (time.perf_counter() - t0) * 1000.0\n\n        if len(results) != len(batch):\n            raise RuntimeError(\n                f"YOLO result count mismatch: {len(results)} vs {len(batch)}"\n            )\n\n        batch_rows = []\n\n        for r, result in zip(batch, results):\n            if result.probs is None:\n                raise RuntimeError(\n                    "YOLO classification result has no probs."\n                )\n\n            yolo_idx = int(result.probs.top1)\n            confidence = float(result.probs.top1conf.item())\n\n            pred_label = canonical_label(\n                model.names[yolo_idx]\n            )\n\n            mapped_idx = class_to_idx.get(pred_label.lower())\n\n            if mapped_idx is None:\n                raise RuntimeError(\n                    f"Cannot map YOLO class: {pred_label}"\n                )\n\n            pred_label = CLASS_LABELS[mapped_idx]\n\n            batch_rows.append({\n                "relative_path": r["relative_path"],\n                "absolute_path": r["absolute_path"],\n                "true_label": r["label"],\n                "true_index": int(r["true_index"]),\n                "pred_index": int(mapped_idx),\n                "pred_label": pred_label,\n                "confidence": confidence,\n                "correct": pred_label == r["label"],\n                "batch_elapsed_ms": float(elapsed_ms),\n                "batch_mean_inference_ms_per_image": float(\n                    elapsed_ms / max(1, len(batch))\n                ),\n            })\n\n        checkpoint(batch_rows)\n\n        print(\n            MODEL_NAME,\n            min(start + len(batch), len(records)),\n            "/",\n            len(records),\n        )\n\n# ------------------------------------------------------------\n# Finalize\n# ------------------------------------------------------------\ndf = pd.DataFrame(rows)\n\nif len(df) != len(manifest):\n    raise RuntimeError(\n        f"Expected {len(manifest)} predictions, got {len(df)}"\n    )\n\nif df["relative_path"].nunique() != len(manifest):\n    raise RuntimeError(\n        "Duplicate or missing prediction rows."\n    )\n\ndf.to_csv(final_csv, index=False)\n\nif partial_csv.exists():\n    partial_csv.unlink()\n\nmetadata = {\n    "model": MODEL_NAME,\n    "framework": framework,\n    "load_seconds": load_seconds,\n    "n_predictions": int(len(df)),\n    "img_size": IMG_SIZE,\n    "batch_size": BATCH_SIZE,\n    "effnet_runtime_compatibility": (\n        "EffNetPreprocessCompat replacing only serialized no-weight "\n        "effnet_preprocess Lambda; same preprocess_input(x*255.0); weights unchanged."\n        if MODEL_NAME == "EfficientNetB0"\n        else None\n    ),\n}\n\nmeta_json.write_text(\n    json.dumps(metadata, indent=2),\n    encoding="utf-8",\n)\n\nprint("__PASS__")\nprint(json.dumps(metadata))\n',
    encoding="utf-8"
)


## Sequential classification evaluation — benchmark + cross-domain

Runs all five classification checkpoints sequentially on both final evaluation datasets.


In [ ]:
# ============================================================
# 8. SEQUENTIAL CLASSIFICATION
# ============================================================

run_status = []

for dataset_name in [
    "benchmark",
    "cross_domain",
]:
    print(
        "\n\n"
        + "#"
        * 90
    )

    print(
        "DATASET:",
        dataset_name
    )

    print(
        "#"
        * 90
    )

    manifest_csv = dataset_info[
        dataset_name
    ][
        "manifest_csv"
    ]

    for model_name in CLASSIFICATION_MODELS:
        out_dir = (
            OUTPUT_ROOT
            / "classification"
            / dataset_name
            / model_name
        )

        final_csv = (
            out_dir
            / "predictions.csv"
        )

        if final_csv.exists():
            print(
                "SKIP completed:",
                dataset_name,
                model_name
            )

            run_status.append({
                "dataset":
                    dataset_name,

                "model":
                    model_name,

                "returncode":
                    0,

                "status":
                    "PASS_CACHED",

                "wall_seconds":
                    None,
            })

            continue

        cmd = [
            sys.executable,
            str(CLS_WORKER_PATH),
            model_name,
            str(
                MODEL_PATHS[
                    model_name
                ]
            ),
            str(manifest_csv),
            str(CLASS_MAPPING_CSV),
            str(out_dir),
            str(IMG_SIZE),
            str(
                BATCH_SIZE[
                    model_name
                ]
            ),
        ]

        result = run_child(
            f"{dataset_name}::{model_name}",
            cmd,
            timeout=8*60*60,
        )

        run_status.append({
            "dataset":
                dataset_name,

            "model":
                model_name,

            "returncode":
                result[
                    "returncode"
                ],

            "status":
                (
                    "PASS"
                    if result[
                        "returncode"
                    ] == 0
                    else
                    "FAILED"
                ),

            "wall_seconds":
                result[
                    "wall_seconds"
                ],
        })

status_df = pd.DataFrame(
    run_status
)

status_df.to_csv(
    OUTPUT_ROOT
    / "classification_worker_status.csv",
    index=False
)

display(status_df)


## 10. Aggregate metrics, per-class evidence, and confusion matrices


In [ ]:
# ============================================================
# 9. AGGREGATE CLASSIFICATION RESULTS
# ============================================================

summary_rows = []
cluster_rows = []

for dataset_name in [
    "benchmark",
    "cross_domain",
]:
    manifest = dataset_info[
        dataset_name
    ]["manifest"]

    for model_name in CLASSIFICATION_MODELS:
        out_dir = (
            OUTPUT_ROOT
            / "classification"
            / dataset_name
            / model_name
        )

        pred_csv = (
            out_dir
            / "predictions.csv"
        )

        if not pred_csv.exists():
            summary_rows.append({
                "dataset":
                    dataset_name,

                "model":
                    model_name,

                "status":
                    "FAILED_OR_INCOMPLETE",
            })

            continue

        df = pd.read_csv(
            pred_csv
        )

        if len(df) != len(manifest):
            summary_rows.append({
                "dataset":
                    dataset_name,

                "model":
                    model_name,

                "status":
                    "INCOMPLETE",

                "n_predictions":
                    int(len(df)),

                "expected":
                    int(len(manifest)),
            })

            continue

        y_true = (
            df["true_label"]
            .astype(str)
            .tolist()
        )

        y_pred = (
            df["pred_label"]
            .astype(str)
            .tolist()
        )

        metrics = {
            "dataset":
                dataset_name,

            "model":
                model_name,

            "status":
                "PASS",

            "n_images":
                int(len(df)),

            "accuracy":
                float(
                    accuracy_score(
                        y_true,
                        y_pred
                    )
                ),

            "macro_precision":
                float(
                    precision_score(
                        y_true,
                        y_pred,
                        labels=CLASS_LABELS,
                        average="macro",
                        zero_division=0,
                    )
                ),

            "macro_recall":
                float(
                    recall_score(
                        y_true,
                        y_pred,
                        labels=CLASS_LABELS,
                        average="macro",
                        zero_division=0,
                    )
                ),

            "macro_f1":
                float(
                    f1_score(
                        y_true,
                        y_pred,
                        labels=CLASS_LABELS,
                        average="macro",
                        zero_division=0,
                    )
                ),

            "weighted_f1":
                float(
                    f1_score(
                        y_true,
                        y_pred,
                        labels=CLASS_LABELS,
                        average="weighted",
                        zero_division=0,
                    )
                ),

            "mean_confidence":
                float(
                    df[
                        "confidence"
                    ].mean()
                ),
        }

        summary_rows.append(
            metrics
        )

        save_json(
            metrics,
            out_dir
            / "metrics_summary.json"
        )

        report = classification_report(
            y_true,
            y_pred,
            labels=CLASS_LABELS,
            output_dict=True,
            zero_division=0,
        )

        pd.DataFrame(
            report
        ).T.to_csv(
            out_dir
            / "classification_report.csv"
        )

        cm = confusion_matrix(
            y_true,
            y_pred,
            labels=CLASS_LABELS,
        )

        cm_df = pd.DataFrame(
            cm,
            index=CLASS_LABELS,
            columns=CLASS_LABELS,
        )

        cm_df.to_csv(
            out_dir
            / "confusion_matrix.csv"
        )

        fig = plt.figure(
            figsize=(14,14)
        )

        ax = fig.add_subplot(111)

        im = ax.imshow(
            cm,
            interpolation="nearest",
            cmap="Blues",
        )

        # Keep the same blue confusion-matrix style as the manuscript.
        display_name = {
            "Custom_CNN": "Custom CNN",
            "EfficientNetB0": "EfficientNetB0",
            "ResNet50V2": "ResNet50V2",
            "ViT": "ViT",
            "YOLOv11m_cls": "YOLOv11m-cls",
        }.get(model_name, model_name)

        ax.set_title(
            display_name,
            fontsize=15,
            pad=8,
        )

        ax.set_xlabel(
            "Predicted"
        )

        ax.set_ylabel(
            "True"
        )

        ax.set_xticks(
            range(
                len(CLASS_LABELS)
            )
        )

        ax.set_yticks(
            range(
                len(CLASS_LABELS)
            )
        )

        ax.set_xticklabels(
            CLASS_LABELS,
            rotation=90,
            fontsize=12.5,
        )

        ax.set_yticklabels(
            CLASS_LABELS,
            fontsize=12.5,
        )

        # Print integer counts in every cell, matching the
        # confusion-matrix presentation used in the paper.
        threshold = cm.max() / 2.0 if cm.size else 0

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                value = int(cm[i, j])

                ax.text(
                    j,
                    i,
                    str(value),
                    ha="center",
                    va="center",
                    fontsize=15,
                    color=(
                        "white"
                        if value > threshold
                        else "black"
                    ),
                )

        cbar = fig.colorbar(
            im,
            ax=ax,
            fraction=0.046,
            pad=0.04,
        )
        cbar.ax.tick_params(labelsize=12.5)

        fig.tight_layout(pad=0.6)

        fig.savefig(
            out_dir
            / "confusion_matrix.png",
            dpi=300,
            bbox_inches="tight",
        )

        plt.close(fig)

        errors = df[
            df["true_label"]
            !=
            df["pred_label"]
        ].copy()

        (
            errors.groupby(
                [
                    "true_label",
                    "pred_label",
                ]
            )
            .size()
            .rename("count")
            .reset_index()
            .sort_values(
                "count",
                ascending=False
            )
            .to_csv(
                out_dir
                / "confusion_pairs.csv",
                index=False
            )
        )

        cluster_samples = df[
            df[
                "true_label"
            ].isin(AEMNST)
        ]

        cluster_errors = cluster_samples[
            cluster_samples[
                "true_label"
            ]
            !=
            cluster_samples[
                "pred_label"
            ]
        ]

        within_cluster = errors[
            errors[
                "true_label"
            ].isin(AEMNST)
            &
            errors[
                "pred_label"
            ].isin(AEMNST)
        ]

        cluster_row = {
            "dataset":
                dataset_name,

            "model":
                model_name,

            "total_errors":
                int(len(errors)),

            "cluster_samples":
                int(
                    len(
                        cluster_samples
                    )
                ),

            "cluster_errors":
                int(
                    len(
                        cluster_errors
                    )
                ),

            "cluster_error_rate":
                (
                    float(
                        len(cluster_errors)
                        /
                        len(cluster_samples)
                    )
                    if len(
                        cluster_samples
                    )
                    else None
                ),

            "cluster_share_of_all_errors":
                (
                    float(
                        len(cluster_errors)
                        /
                        len(errors)
                    )
                    if len(errors)
                    else 0.0
                ),

            "within_cluster_confusions":
                int(
                    len(
                        within_cluster
                    )
                ),

            "within_cluster_share_of_all_errors":
                (
                    float(
                        len(within_cluster)
                        /
                        len(errors)
                    )
                    if len(errors)
                    else 0.0
                ),
        }

        cluster_rows.append(
            cluster_row
        )

classification_summary = pd.DataFrame(
    summary_rows
)

classification_summary.to_csv(
    OUTPUT_ROOT
    / "classification_summary_benchmark_cross.csv",
    index=False
)

cluster_summary = pd.DataFrame(
    cluster_rows
)

cluster_summary.to_csv(
    OUTPUT_ROOT
    / "AEMNST_summary_benchmark_cross.csv",
    index=False
)

display(classification_summary)

display(cluster_summary)


## 10A. Figure 4 — Vision Transformer confusion matrix on the final cross-domain set

This cell regenerates and **displays** the paper-facing confusion matrix for the Vision Transformer using the fresh `cross_domain/ViT/predictions.csv` produced above. It also saves the matrix as CSV and PNG for manuscript/evidence use.


In [ ]:
# ============================================================
# 10A. FIGURE 4 — VISION TRANSFORMER CROSS-DOMAIN CONFUSION MATRIX
# ============================================================
# Run this cell AFTER the sequential classification / aggregation cells.
# It uses the fresh predictions from the final 870-image cross-domain set.

vit_pred_csv = (
    OUTPUT_ROOT
    / "classification"
    / "cross_domain"
    / "ViT"
    / "predictions.csv"
)

if not vit_pred_csv.exists():
    raise FileNotFoundError(
        "ViT cross-domain predictions.csv was not found. "
        "Run the sequential classification cells first."
    )

vit_df = pd.read_csv(vit_pred_csv)

required_cols = {"true_label", "pred_label"}
missing_cols = required_cols - set(vit_df.columns)
if missing_cols:
    raise RuntimeError(
        f"Missing required prediction columns: {sorted(missing_cols)}"
    )

# Canonicalize labels to keep the same semantic class naming as the notebook.
vit_df["true_label"] = vit_df["true_label"].astype(str).map(canonical_label)
vit_df["pred_label"] = vit_df["pred_label"].astype(str).map(canonical_label)

if len(vit_df) != CROSS_DOMAIN_EXPECTED_TOTAL:
    raise RuntimeError(
        f"Expected {CROSS_DOMAIN_EXPECTED_TOTAL} cross-domain predictions, "
        f"but found {len(vit_df)}."
    )

unknown_true = sorted(set(vit_df["true_label"]) - set(CLASS_LABELS))
unknown_pred = sorted(set(vit_df["pred_label"]) - set(CLASS_LABELS))

if unknown_true or unknown_pred:
    raise RuntimeError(
        "Label mismatch detected. "
        f"unknown_true={unknown_true}, unknown_pred={unknown_pred}"
    )

y_true = vit_df["true_label"].tolist()
y_pred = vit_df["pred_label"].tolist()

vit_cm = confusion_matrix(
    y_true,
    y_pred,
    labels=CLASS_LABELS,
)

vit_acc = accuracy_score(y_true, y_pred)

# Save the numeric matrix for editorial evidence.
vit_cm_csv = (
    OUTPUT_ROOT
    / "classification"
    / "cross_domain"
    / "ViT"
    / "Figure4_ViT_cross_domain_confusion_matrix.csv"
)

pd.DataFrame(
    vit_cm,
    index=CLASS_LABELS,
    columns=CLASS_LABELS,
).to_csv(vit_cm_csv)

# Paper-facing figure.
fig, ax = plt.subplots(figsize=(13.2, 10.0))

im = ax.imshow(
    vit_cm,
    interpolation="nearest",
    cmap="Blues",
)

ax.set_title("ViT", fontsize=18, pad=8)
ax.set_xlabel("Predicted", fontsize=14)
ax.set_ylabel("True", fontsize=14)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(axis="both", which="both", length=0)
ax.set_aspect("equal", adjustable="box")

ax.set_xticks(range(len(CLASS_LABELS)))
ax.set_yticks(range(len(CLASS_LABELS)))

ax.set_xticklabels(
    CLASS_LABELS,
    rotation=90,
    fontsize=12.5,
)
ax.set_yticklabels(
    CLASS_LABELS,
    fontsize=12.5,
)

# Print integer counts in every cell, matching the paper-style matrix.
threshold = vit_cm.max() / 2.0 if vit_cm.size else 0

for i in range(vit_cm.shape[0]):
    for j in range(vit_cm.shape[1]):
        value = int(vit_cm[i, j])
        ax.text(
            j,
            i,
            str(value),
            ha="center",
            va="center",
            fontsize=15,
            color=("white" if value > threshold else "black"),
        )

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=12.5)
fig.tight_layout(pad=0.6)

vit_cm_png = (
    OUTPUT_ROOT
    / "classification"
    / "cross_domain"
    / "ViT"
    / "Figure4_ViT_cross_domain_confusion_matrix.png"
)

fig.savefig(
    vit_cm_png,
    dpi=300,
    bbox_inches="tight",
)

print("=" * 80)
print("FIGURE 4 — VISION TRANSFORMER / CROSS-DOMAIN")
print("=" * 80)
print(f"Images evaluated : {len(vit_df)}")
print(f"Accuracy         : {vit_acc * 100:.2f}%")
print(f"PNG saved to     : {vit_cm_png}")
print(f"CSV saved to     : {vit_cm_csv}")
print("=" * 80)

plt.show()


## 11. Original published values versus post-publication rerun


In [ ]:
# ============================================================
# 10. PUBLISHED vs RERUN + ACCURACY DEGRADATION
# ============================================================

PUBLISHED_CLASSIFICATION = {
    "Custom_CNN":{
        "benchmark_accuracy":0.9997,
        "cross_accuracy":0.4592,
        "cross_macro_precision":0.6769,
        "cross_macro_recall":0.4590,
        "cross_macro_f1":0.4993,
    },

    "EfficientNetB0":{
        "benchmark_accuracy":0.9999,
        "cross_accuracy":0.7382,
        "cross_macro_precision":0.8235,
        "cross_macro_recall":0.7383,
        "cross_macro_f1":0.7408,
    },

    "ResNet50V2":{
        "benchmark_accuracy":0.9998,
        "cross_accuracy":0.6154,
        "cross_macro_precision":0.7501,
        "cross_macro_recall":0.6155,
        "cross_macro_f1":0.6273,
    },

    "ViT":{
        "benchmark_accuracy":0.9994,
        "cross_accuracy":0.8393,
        "cross_macro_precision":0.8629,
        "cross_macro_recall":0.8394,
        "cross_macro_f1":0.8354,
    },

    "YOLOv11m_cls":{
        "benchmark_accuracy":0.9999,
        "cross_accuracy":0.6877,
        "cross_macro_precision":0.7821,
        "cross_macro_recall":0.6878,
        "cross_macro_f1":0.6907,
    },
}

comparison_rows = []
degradation_rows = []

for model_name in CLASSIFICATION_MODELS:
    bench_row = classification_summary[
        (
            classification_summary[
                "dataset"
            ]=="benchmark"
        )
        &
        (
            classification_summary[
                "model"
            ]==model_name
        )
        &
        (
            classification_summary[
                "status"
            ]=="PASS"
        )
    ]

    cross_row = classification_summary[
        (
            classification_summary[
                "dataset"
            ]=="cross_domain"
        )
        &
        (
            classification_summary[
                "model"
            ]==model_name
        )
        &
        (
            classification_summary[
                "status"
            ]=="PASS"
        )
    ]

    pub = PUBLISHED_CLASSIFICATION[
        model_name
    ]

    if not bench_row.empty:
        b = bench_row.iloc[0]

        comparison_rows.append({
            "dataset":
                "benchmark",

            "model":
                model_name,

            "published_accuracy":
                pub[
                    "benchmark_accuracy"
                ],

            "rerun_accuracy":
                b[
                    "accuracy"
                ],

            "accuracy_diff_pp":
                100.0
                * (
                    b[
                        "accuracy"
                    ]
                    -
                    pub[
                        "benchmark_accuracy"
                    ]
                ),
        })

    if not cross_row.empty:
        c = cross_row.iloc[0]

        comparison_rows.append({
            "dataset":
                "cross_domain",

            "model":
                model_name,

            "published_accuracy":
                pub[
                    "cross_accuracy"
                ],

            "rerun_accuracy":
                c[
                    "accuracy"
                ],

            "accuracy_diff_pp":
                100.0
                * (
                    c[
                        "accuracy"
                    ]
                    -
                    pub[
                        "cross_accuracy"
                    ]
                ),

            "published_macro_precision":
                pub[
                    "cross_macro_precision"
                ],

            "rerun_macro_precision":
                c[
                    "macro_precision"
                ],

            "published_macro_recall":
                pub[
                    "cross_macro_recall"
                ],

            "rerun_macro_recall":
                c[
                    "macro_recall"
                ],

            "published_macro_f1":
                pub[
                    "cross_macro_f1"
                ],

            "rerun_macro_f1":
                c[
                    "macro_f1"
                ],
        })

    if (
        not bench_row.empty
        and
        not cross_row.empty
    ):
        b = bench_row.iloc[0]
        c = cross_row.iloc[0]

        degradation_rows.append({
            "model":
                model_name,

            "metric":
                "accuracy",

            "benchmark":
                b[
                    "accuracy"
                ],

            "cross_domain":
                c[
                    "accuracy"
                ],

            "drop_pp":
                100.0
                * (
                    b[
                        "accuracy"
                    ]
                    -
                    c[
                        "accuracy"
                    ]
                ),
        })

published_vs_rerun = pd.DataFrame(
    comparison_rows
)

published_vs_rerun.to_csv(
    OUTPUT_ROOT
    / "published_vs_rerun_classification.csv",
    index=False
)

classification_degradation = pd.DataFrame(
    degradation_rows
)

classification_degradation.to_csv(
    OUTPUT_ROOT
    / "classification_benchmark_to_cross_degradation.csv",
    index=False
)

display(published_vs_rerun)

display(classification_degradation)


In [ ]:
# ============================================================
# 11. EDITOR-FACING CONSISTENCY CHECKS
#     AEMNST concentration + corrected manuscript values
# ============================================================

aemnst_rows = []

for model_name in CLASSIFICATION_MODELS:
    pred_csv = (
        WORK_ROOT / "classification" / "cross_domain"
        / model_name / "predictions.csv"
    )
    df = pd.read_csv(pred_csv)

    in_cluster = df["true_label"].isin(AEMNST)
    err = df["true_label"] != df["pred_label"]

    cluster_n = int(in_cluster.sum())
    cluster_err = int((in_cluster & err).sum())

    noncluster_n = int((~in_cluster).sum())
    noncluster_err = int(((~in_cluster) & err).sum())

    cluster_rate = cluster_err / cluster_n if cluster_n else None
    noncluster_rate = noncluster_err / noncluster_n if noncluster_n else None

    aemnst_rows.append({
        "model": model_name,
        "cluster_samples": cluster_n,
        "cluster_errors": cluster_err,
        "cluster_error_rate": cluster_rate,
        "noncluster_samples": noncluster_n,
        "noncluster_errors": noncluster_err,
        "noncluster_error_rate": noncluster_rate,
        "cluster_to_noncluster_error_rate_ratio": (
            cluster_rate / noncluster_rate
            if noncluster_rate not in (None, 0)
            else None
        ),
        "cluster_error_rate_elevated": (
            bool(cluster_rate > noncluster_rate)
            if cluster_rate is not None and noncluster_rate is not None
            else None
        ),
    })

aemnst_detailed = pd.DataFrame(aemnst_rows)
aemnst_detailed.to_csv(
    WORK_ROOT / "AEMNST_detailed_cross_domain.csv",
    index=False
)

display(aemnst_detailed)

# Values currently used in the corrected manuscript (percent units).
CORRECTED_MANUSCRIPT = {
    "Custom_CNN":     {"benchmark": 99.95, "cross_domain": 43.33},
    "EfficientNetB0": {"benchmark": 100.00, "cross_domain": 71.61},
    "ResNet50V2":     {"benchmark": 99.99, "cross_domain": 59.89},
    "ViT":            {"benchmark": 99.94, "cross_domain": 83.45},
    "YOLOv11m_cls":   {"benchmark": 99.21, "cross_domain": 51.03},
}

consistency_rows = []

for model_name in CLASSIFICATION_MODELS:
    for dataset_name in ["benchmark", "cross_domain"]:
        row = classification_summary[
            (classification_summary["model"] == model_name)
            & (classification_summary["dataset"] == dataset_name)
            & (classification_summary["status"] == "PASS")
        ]

        if row.empty:
            continue

        rerun_pct = 100.0 * float(row.iloc[0]["accuracy"])
        expected_pct = CORRECTED_MANUSCRIPT[model_name][dataset_name]
        diff_pp = rerun_pct - expected_pct

        consistency_rows.append({
            "dataset": dataset_name,
            "model": model_name,
            "corrected_manuscript_accuracy_pct": expected_pct,
            "rerun_accuracy_pct": rerun_pct,
            "difference_pp": diff_pp,
            "within_0_05_pp": abs(diff_pp) <= 0.05,
        })

manuscript_consistency = pd.DataFrame(consistency_rows)
manuscript_consistency.to_csv(
    WORK_ROOT / "corrected_manuscript_consistency_check.csv",
    index=False
)

display(manuscript_consistency)

if not manuscript_consistency["within_0_05_pp"].all():
    print(
        "WARNING: At least one rerun differs by more than 0.05 pp "
        "from the rounded value currently used in the corrected manuscript."
    )
else:
    print("✅ All corrected-manuscript classification values match within 0.05 pp.")


## Build the classification-only editor-facing evidence package


In [ ]:
# ============================================================
# BUILD SANITIZED CLASSIFICATION-ONLY EDITOR EVIDENCE PACKAGE
# ============================================================

if EVIDENCE_ROOT.exists():
    shutil.rmtree(EVIDENCE_ROOT)

EVIDENCE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

def sanitize_csv(src_path, dst_path):
    src_path = Path(src_path)
    dst_path = Path(dst_path)

    dst_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    df = pd.read_csv(src_path)

    drop_cols = [
        c for c in df.columns
        if c.lower() in {
            "absolute_path",
            "path",
            "compat_model_path",
        }
    ]

    if drop_cols:
        df = df.drop(
            columns=drop_cols
        )

    df.to_csv(
        dst_path,
        index=False,
    )

# ------------------------------------------------------------
# A. Protocol / environment
# ------------------------------------------------------------
environment = {
    "purpose":
        "post-publication technical re-examination — classification branch",

    "evaluation_scope":
        "classification only",

    "python":
        sys.version,

    "platform":
        platform.platform(),

    "gpu":
        TEST_GPU_NAME,

    "seed":
        SEED,

    "benchmark_dataset_slug":
        "toilaxien/asl-benchmark-test-split",

    "cross_domain_dataset_slug":
        "toilaxien/asl-test-cross-domain-datase/cross_domain_dataset",

    "model_dataset_slug":
        "toilaxien/asl-models-realtime-test",

    "benchmark_images_used":
        int(
            len(
                dataset_info[
                    "benchmark"
                ][
                    "manifest"
                ]
            )
        ),

    "cross_domain_images_used":
        int(
            len(
                dataset_info[
                    "cross_domain"
                ][
                    "manifest"
                ]
            )
        ),

    "cross_domain_protocol":
        "29 classes × 30 images/class = 870",

    "img_size":
        IMG_SIZE,

    "class_order":
        CLASS_LABELS,

    "classification_models":
        CLASSIFICATION_MODELS,

    "vit_preprocessing":
        (
            "Resize 224x224; ToTensor; "
            "Normalize mean=(0.5,0.5,0.5), "
            "std=(0.5,0.5,0.5)"
        ),

    "yolov11m_cls_label_alias":
        "del -> delete",

    "efficientnet_runtime_note":
        (
            "Keras-3 runtime compatibility reconstruction replaces only "
            "the serialized no-weight effnet_preprocess Lambda with "
            "EffNetPreprocessCompat; model weights are unchanged."
        ),

    "detection_note":
        "Detection evaluation is intentionally excluded from this notebook.",
}

save_json(
    environment,
    EVIDENCE_ROOT / "environment_and_protocol.json",
)

# ------------------------------------------------------------
# B. Dataset evidence
# ------------------------------------------------------------
for dataset_name in [
    "benchmark",
    "cross_domain",
]:
    src_dir = (
        WORK_ROOT
        / "datasets"
        / dataset_name
    )

    dst_dir = (
        EVIDENCE_ROOT
        / "datasets"
        / dataset_name
    )

    dst_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    for csv_path in src_dir.glob(
        "*.csv"
    ):
        sanitize_csv(
            csv_path,
            dst_dir / csv_path.name,
        )

    for json_path in src_dir.glob(
        "*.json"
    ):
        data = json.loads(
            json_path.read_text(
                encoding="utf-8"
            )
        )

        data.pop(
            "resolved_class_root",
            None,
        )

        save_json(
            data,
            dst_dir / json_path.name,
        )

overlap_csv = (
    WORK_ROOT
    / "benchmark_cross_exact_overlap.csv"
)

if overlap_csv.exists():
    sanitize_csv(
        overlap_csv,
        EVIDENCE_ROOT
        / "datasets"
        / "benchmark_cross_exact_overlap.csv",
    )

sanitize_csv(
    WORK_ROOT / "class_mapping.csv",
    EVIDENCE_ROOT / "class_mapping.csv",
)

# ------------------------------------------------------------
# C. Classification model inventory
# ------------------------------------------------------------
sanitize_csv(
    WORK_ROOT / "model_inventory.csv",
    EVIDENCE_ROOT / "model_inventory.csv",
)

eff_audit_path = (
    WORK_ROOT
    / "efficientnet_compatibility_audit.json"
)

if eff_audit_path.exists():
    eff_audit = json.loads(
        eff_audit_path.read_text(
            encoding="utf-8"
        )
    )

    eff_audit.pop(
        "compat_model_path",
        None,
    )

    save_json(
        eff_audit,
        EVIDENCE_ROOT
        / "efficientnet_compatibility_audit.json",
    )

# ------------------------------------------------------------
# D. Classification summary files only
# ------------------------------------------------------------
summary_files = [
    "classification_summary_benchmark_cross.csv",
    "AEMNST_summary_benchmark_cross.csv",
    "AEMNST_detailed_cross_domain.csv",
    "published_vs_rerun_classification.csv",
    "classification_benchmark_to_cross_degradation.csv",
    "corrected_manuscript_consistency_check.csv",
    "classification_worker_status.csv",
]

for name in summary_files:
    p = WORK_ROOT / name

    if p.exists():
        sanitize_csv(
            p,
            EVIDENCE_ROOT / name,
        )

# ------------------------------------------------------------
# E. Per-model raw predictions / reports / matrices
# ------------------------------------------------------------
for dataset_name in [
    "benchmark",
    "cross_domain",
]:
    for model_name in CLASSIFICATION_MODELS:
        src_dir = (
            WORK_ROOT
            / "classification"
            / dataset_name
            / model_name
        )

        if not src_dir.exists():
            continue

        dst_dir = (
            EVIDENCE_ROOT
            / "classification"
            / dataset_name
            / model_name
        )

        dst_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        for p in src_dir.iterdir():
            if not p.is_file():
                continue

            if p.suffix.lower() == ".csv":
                sanitize_csv(
                    p,
                    dst_dir / p.name,
                )

            elif p.suffix.lower() in {
                ".json",
                ".png",
            }:
                shutil.copy2(
                    p,
                    dst_dir / p.name,
                )

# ------------------------------------------------------------
# F. Classification reproducibility code
# ------------------------------------------------------------
code_dir = (
    EVIDENCE_ROOT
    / "code"
)

code_dir.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copy2(
    CLS_WORKER_PATH,
    code_dir / CLS_WORKER_PATH.name,
)

# ------------------------------------------------------------
# G. Environment freeze
# ------------------------------------------------------------
pip_freeze = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "freeze",
    ],
    capture_output=True,
    text=True,
    check=False,
)

(
    EVIDENCE_ROOT
    / "pip_freeze.txt"
).write_text(
    pip_freeze.stdout,
    encoding="utf-8",
)

# ------------------------------------------------------------
# H. README
# ------------------------------------------------------------
readme = f"""
# IJIES Classification Benchmark + Cross-domain Re-evaluation Evidence

This package contains the post-publication technical re-examination
of the classification branch only.

## Evaluation sets
- Benchmark test: {len(dataset_info["benchmark"]["manifest"])} images,
  29 classes, 600 images/class.
- Cross-domain: {len(dataset_info["cross_domain"]["manifest"])} images,
  29 classes, 30 images/class.

The notebook requires exactly 870 cross-domain images and audits exact
SHA256 overlap against the benchmark test set.

## Classification models
1. Custom CNN
2. EfficientNetB0
3. ResNet50V2
4. Vision Transformer (ViT)
5. YOLOv11m-cls

## Evaluation outputs
For each model/dataset:
- predictions.csv
- metrics_summary.json
- classification_report.csv
- confusion_matrix.csv
- confusion_matrix.png
- confusion_pairs.csv

The package also contains benchmark-to-cross-domain degradation,
A/E/M/N/S/T analysis, model checkpoint hashes, dataset manifests,
and the classification worker code.

## Scope
Detection models and detection mAP evaluation are intentionally excluded.
"""

(
    EVIDENCE_ROOT / "README.md"
).write_text(
    textwrap.dedent(readme).strip() + "\n",
    encoding="utf-8",
)

# ------------------------------------------------------------
# I. ZIP + convenient summary copies
# ------------------------------------------------------------
archive = shutil.make_archive(
    "/kaggle/working/"
    "IJIES_EDITOR_CLASSIFICATION_BENCHMARK_CROSSDOMAIN_EVIDENCE",
    "zip",
    root_dir=str(EVIDENCE_ROOT),
)

shutil.copy2(
    WORK_ROOT
    / "classification_summary_benchmark_cross.csv",
    "/kaggle/working/"
    "IJIES_EDITOR_CLASSIFICATION_BENCHMARK_CROSS_FINAL.csv",
)

shutil.copy2(
    WORK_ROOT
    / "classification_benchmark_to_cross_degradation.csv",
    "/kaggle/working/"
    "IJIES_EDITOR_CLASSIFICATION_DEGRADATION_FINAL.csv",
)

print(
    "✅ Classification-only evidence folder:",
    EVIDENCE_ROOT,
)

print(
    "✅ Classification-only evidence ZIP:",
    archive,
)

print(
    "✅ Final classification summary:",
    "/kaggle/working/"
    "IJIES_EDITOR_CLASSIFICATION_BENCHMARK_CROSS_FINAL.csv",
)

print(
    "✅ Final degradation summary:",
    "/kaggle/working/"
    "IJIES_EDITOR_CLASSIFICATION_DEGRADATION_FINAL.csv",
)


### Files to place in the public repository

After a successful run, use `IJIES_EDITOR_CLASSIFICATION_BENCHMARK_CROSSDOMAIN_EVIDENCE.zip` as the canonical classification evidence package.
Do not mix it with older 871-image outputs or detector evidence.
